In [2]:
!pip install numpy scipy casadi pyserial matplotlib pyserial
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import casadi as ca
import time

# Parámetros nominales de la planta (los mismos de tu código)
P_NOM = {'Atank': 3.02e-3, 'rho': 997.0, 'eta': 8.9e-4, 'g': 9.81,
         'alphao1': 0.0271, 'Do1': 10e-3, 'alphao2': 0.0271, 'A2': 7.85e-5, 'alphao3': 0.0271, 'Do3': 10e-3,
         'alpha120': 0.3038, 'D12': 10e-3, 'A12': 7.85e-5, 'lambdac12': 24000,
         'alpha230': 0.1344, 'D23': 10e-3, 'A23': 7.85e-5, 'lambdac23': 29600,
         'qi1max': 2.69e-5, 'qi3max': 2.41e-5}
Ts = 2.0
EPS = 1e-10

# 1. Definir la ODE en CasADi
x_sym = ca.SX.sym('x', 3)
u_sym = ca.SX.sym('u', 2)

h1, h2, h3 = x_sym[0], x_sym[1], x_sym[2]
qi1 = P_NOM['qi1max'] * u_sym[0]
qi3 = P_NOM['qi3max'] * u_sym[1]
qo1 = P_NOM['alphao1'] * (P_NOM['Do1']**2 * np.pi / 4) * ca.sqrt(2 * P_NOM['g'] * h1 + EPS)
qo2 = P_NOM['alphao2'] * P_NOM['A2'] * ca.sqrt(2 * P_NOM['g'] * h2 + EPS)
qo3 = P_NOM['alphao3'] * (P_NOM['Do3']**2 * np.pi / 4) * ca.sqrt(2 * P_NOM['g'] * h3 + EPS)

dh12 = h1 - h2
l12 = P_NOM['D12'] * P_NOM['rho'] / P_NOM['eta'] * ca.sqrt(2 * P_NOM['g'] * ca.fabs(dh12) + EPS)
a12 = P_NOM['alpha120'] * ca.tanh(2 * l12 / P_NOM['lambdac12'])
q12 = a12 * P_NOM['A12'] * ca.sqrt(2 * P_NOM['g'] * ca.fabs(dh12) + EPS) * ca.sign(dh12)

dh23 = h2 - h3
l23 = P_NOM['D23'] * P_NOM['rho'] / P_NOM['eta'] * ca.sqrt(2 * P_NOM['g'] * ca.fabs(dh23) + EPS)
a23 = P_NOM['alpha230'] * ca.tanh(2 * l23 / P_NOM['lambdac23'])
q23 = a23 * P_NOM['A23'] * ca.sqrt(2 * P_NOM['g'] * ca.fabs(dh23) + EPS) * ca.sign(dh23)

dx = ca.vertcat((qi1 - q12 - qo1) / P_NOM['Atank'],
                (q12 - q23 - qo2) / P_NOM['Atank'],
                (qi3 + q23 - qo3) / P_NOM['Atank'])

# Jacobiano analítico de f respecto a x
J_sym = ca.jacobian(dx, x_sym)
# Jacobiano discreto (Euler Fk = I + Ts * J)
Fk_sym = ca.SX.eye(3) + Ts * J_sym
calc_Fk = ca.Function('calc_Fk', [x_sym, u_sym], [Fk_sym])

# 2. Prueba de Velocidad
x_test = [0.15, 0.15, 0.15]
u_test = [0.5, 0.5]

# Calentamiento
_ = calc_Fk(x_test, u_test)

N_test = 1000
t0 = time.perf_counter()
for _ in range(N_test):
    F_val = calc_Fk(x_test, u_test)
t1 = time.perf_counter()

print(f"==================================================")
print(f"  TIEMPO DE JACOBIANO AUTOMÁTICO/ANALÍTICO (CasADi)  ")
print(f"==================================================")
print(f"Tiempo total por {N_test} pasos: {(t1 - t0)*1000:.2f} ms")
print(f"Tiempo promedio por paso EKF:    {((t1 - t0)*1000)/N_test:.4f} ms")
print(f"Tiempo antiguo (Diferencias finitas): ~2090.0 ms")
print(f"==================================================")

  TIEMPO DE JACOBIANO AUTOMÁTICO/ANALÍTICO (CasADi)  
Tiempo total por 1000 pasos: 31.59 ms
Tiempo promedio por paso EKF:    0.0316 ms
Tiempo antiguo (Diferencias finitas): ~2090.0 ms
